# DocuMentor 엔진 개발 - 베이스라인 코드

## 과제 목표
이 베이스라인 코드를 기반으로 더 발전된 RAG 시스템을 구현하세요.

## 파이프라인 구성
1. **환경 설정**: 필요한 라이브러리 설치 및 API 키 설정
2. **데이터 로드**: PDF 파일 업로드
3. **파싱**: PyPDF2를 사용한 텍스트 추출
4. **청킹**: RecursiveCharacterTextSplitter를 사용한 텍스트 분할
5. **임베딩**: OpenAI text-embedding-3-large 모델 사용
6. **벡터 DB 구축**: FAISS 인덱스 생성
7. **검색**: 코사인 유사도 기반 검색
8. **응답 생성**: GPT-4를 활용한 최종 답변 생성

## 1. 환경 설정 및 라이브러리 설치

In [11]:
# 필요한 라이브러리 설치
!pip install -q pypdf2 langchain langchain-openai langchain-community faiss-cpu python-dotenv

In [12]:
# 라이브러리 임포트
import os
import json
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np

# PDF 파싱
from PyPDF2 import PdfReader

# 텍스트 분할
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 임베딩 및 LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 벡터 스토어
import faiss

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


In [13]:
# OpenAI API 키 설정
# 구글 코랩: 왼쪽 사이드바의 '🔑' 아이콘 클릭하여 시크릿에 추가
# 로컬: .env 파일에 OPENAI_API_KEY=your-api-key 형태로 저장

try:
    # 구글 코랩 환경
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("API 키 로드 완료 (Colab)")
except:
    # 로컬 환경
    from dotenv import load_dotenv
    load_dotenv()
    print("API 키 로드 완료 (Local)")

# temp 폴더 생성
TEMP_DIR = Path("./temp")
TEMP_DIR.mkdir(exist_ok=True)
print(f"작업 폴더 생성: {TEMP_DIR}")

API 키 로드 완료 (Colab)
작업 폴더 생성: temp


## 2. 데이터 로드

구글 코랩의 경우 왼쪽 사이드바에서 PDF 파일을 업로드하거나,  
샘플 PDF를 다운로드하여 사용할 수 있습니다.

In [14]:
# 구글 코랩에서 파일 업로드
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"업로드 완료: {pdf_path}")
except:
    # 로컬 환경: 파일 경로 직접 지정
    pdf_path = "your_document.pdf"
    print(f"파일 경로: {pdf_path}")

Saving Skywork.pdf to Skywork (3).pdf
업로드 완료: Skywork (3).pdf


## 3. 파싱 (Parsing)

**Input**: PDF 파일 경로  
**Output**: 추출된 텍스트 문자열  
**저장**: `temp/parsed_text.txt`

In [15]:
def parse_pdf(pdf_path: str) -> str:
    """PDF에서 텍스트 추출"""
    reader = PdfReader(pdf_path)
    text = ""

    for page_num, page in enumerate(reader.pages, 1):
        page_text = page.extract_text()
        text += f"\n--- Page {page_num} ---\n{page_text}"

    return text

# PDF 파싱
print("PDF 파싱 중...")
parsed_text = parse_pdf(pdf_path)
print(f"파싱 완료: {len(parsed_text)} 글자")

# 저장
parsed_text_path = TEMP_DIR / "parsed_text.txt"
with open(parsed_text_path, 'w', encoding='utf-8') as f:
    f.write(parsed_text)
print(f"저장: {parsed_text_path}")

# 샘플 출력
print("\n[텍스트 샘플]")
print(parsed_text[:300] + "...")

PDF 파싱 중...
파싱 완료: 50200 글자
저장: temp/parsed_text.txt

[텍스트 샘플]

--- Page 1 ---
arXiv:2504.16656v4  [cs.CV]  6 Jun 2025Skywork R1V2: Multimodal Hybrid Reinforcement
Learning for Reasoning
Chris∗,Yichen Wei∗,Yi Peng ,Xiaokun Wang ,Weijie Qiu ,Wei Shen ,
Tianyidan Xie, Jiangbo Pei, Jianhao Zhang, Yunzhuo Hao, Xuchen Song†,
Yang Liu†, Yahui Zhou
Skywork AI, Kunlun ...


## 4. 청킹 (Chunking)

**Input**: 파싱된 텍스트  
**Output**: 텍스트 청크 리스트  
**저장**: `temp/chunks.json`

In [16]:
def chunk_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """텍스트를 청크로 분할"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_text(text)
    return chunks

# 텍스트 로드
with open(parsed_text_path, 'r', encoding='utf-8') as f:
    parsed_text = f.read()

# 청킹
print("텍스트 청킹 중...")
chunks = chunk_text(parsed_text, chunk_size=1000, chunk_overlap=200)
print(f"청킹 완료: {len(chunks)}개 청크")

# 저장
chunks_path = TEMP_DIR / "chunks.json"
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump({"chunks": chunks}, f, ensure_ascii=False, indent=2)
print(f"저장: {chunks_path}")

# 통계
chunk_lengths = [len(chunk) for chunk in chunks]
print(f"\n[청크 통계]")
print(f"평균 길이: {np.mean(chunk_lengths):.1f} 글자")
print(f"최소 길이: {np.min(chunk_lengths)} 글자")
print(f"최대 길이: {np.max(chunk_lengths)} 글자")

텍스트 청킹 중...
청킹 완료: 64개 청크
저장: temp/chunks.json

[청크 통계]
평균 길이: 951.6 글자
최소 길이: 508 글자
최대 길이: 998 글자


## 5. 임베딩

**Input**: 텍스트 청크 리스트  
**Output**: 임베딩 벡터 배열  
**저장**: `temp/embeddings.npy`

In [17]:
def generate_embeddings(chunks: List[str], batch_size: int = 100) -> np.ndarray:
    """텍스트 청크들의 임베딩 생성"""
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

    embeddings = []
    total = len(chunks)

    print(f"임베딩 생성 중... (총 {total}개 청크)")
    for i in range(0, total, batch_size):
        batch = chunks[i:i+batch_size]
        embeddings.extend(embeddings_model.embed_documents(batch))
        print(f"진행: {min(i+batch_size, total)}/{total}", end='\r')

    print(f"\n임베딩 생성 완료: {total}개 청크")
    return np.array(embeddings, dtype=np.float32)

# 청크 로드
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)["chunks"]

# 임베딩 생성
embeddings = generate_embeddings(chunks, batch_size=100)
print(f"Shape: {embeddings.shape}")
print(f"Dimension: {embeddings.shape[1]}차원")

# 저장
embeddings_path = TEMP_DIR / "embeddings.npy"
np.save(embeddings_path, embeddings)
print(f"저장: {embeddings_path}")

임베딩 생성 중... (총 64개 청크)
진행: 64/64
임베딩 생성 완료: 64개 청크
Shape: (64, 3072)
Dimension: 3072차원
저장: temp/embeddings.npy


## 6. 벡터 DB 구축 (FAISS)

**Input**: 임베딩 벡터 배열  
**Output**: FAISS 인덱스  
**저장**: `temp/faiss_index.bin`

In [18]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """FAISS 인덱스 구축"""
    dimension = embeddings.shape[1]

    # L2 거리 기반 인덱스 생성
    index = faiss.IndexFlatL2(dimension)

    # 정규화 (코사인 유사도)
    faiss.normalize_L2(embeddings)

    # 벡터 추가
    index.add(embeddings)

    return index

# 임베딩 로드
embeddings = np.load(embeddings_path)

# 인덱스 구축
print("FAISS 인덱스 구축 중...")
faiss_index = build_faiss_index(embeddings)
print(f"인덱스 구축 완료")
print(f"총 벡터 수: {faiss_index.ntotal}")
print(f"차원: {faiss_index.d}")

# 저장
faiss_index_path = TEMP_DIR / "faiss_index.bin"
faiss.write_index(faiss_index, str(faiss_index_path))
print(f"저장: {faiss_index_path}")

FAISS 인덱스 구축 중...
인덱스 구축 완료
총 벡터 수: 64
차원: 3072
저장: temp/faiss_index.bin


## 7. 검색 시스템

**Input**: 사용자 쿼리  
**Output**: 관련성 높은 청크들 (top-k)

In [19]:
def search_chunks(query: str, top_k: int = 3) -> List[Tuple[int, float, str]]:
    """
    쿼리에 대해 관련성 높은 청크 검색

    Returns:
        [(청크_인덱스, 유사도, 청크_텍스트), ...]
    """
    # 쿼리 임베딩
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")
    query_embedding = embeddings_model.embed_query(query)
    query_vector = np.array([query_embedding], dtype=np.float32)

    # 정규화
    faiss.normalize_L2(query_vector)

    # FAISS 검색
    distances, indices = faiss_index.search(query_vector, top_k)

    # 유사도 변환 (L2 거리 -> 코사인 유사도)
    similarities = 1 - (distances[0] ** 2) / 2

    # 결과 구성
    results = []
    for idx, similarity in zip(indices[0], similarities):
        chunk_text = chunks[idx]
        results.append((idx, similarity, chunk_text))

    return results

# 데이터 로드
faiss_index = faiss.read_index(str(faiss_index_path))
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)["chunks"]

print("검색 시스템 준비 완료")

검색 시스템 준비 완료


## 8. 검색된 내용 기반의 LLM 응답 생성

**Input**: 사용자 질문  
**Output**: LLM이 생성한 답변

In [20]:
def generate_answer(query: str, top_k: int = 3) -> Dict[str, any]:
    """
    RAG 파이프라인: 검색 + LLM 답변 생성

    Returns:
        {
            'query': 질문,
            'answer': LLM 답변,
            'retrieved_chunks': 검색된 청크들
        }
    """
    # 1. 관련 청크 검색
    print(f"질문: {query}")
    print("\n관련 문서 검색 중...")
    retrieved_chunks = search_chunks(query, top_k=top_k)

    # 검색 결과 출력
    print(f"검색 완료: {len(retrieved_chunks)}개 청크 발견\n")
    for rank, (idx, similarity, chunk) in enumerate(retrieved_chunks, 1):
        print(f"[{rank}위] 청크 #{idx} (유사도: {similarity:.4f})")
        print(f"{chunk[:150]}...\n")

    # 2. 컨텍스트 구성
    context = "\n\n".join([chunk for _, _, chunk in retrieved_chunks])

    # 3. 프롬프트 생성
    prompt = f"""당신은 문서 기반 질의응답 시스템입니다. 주어진 문서 내용을 바탕으로 질문에 답변하세요.

규칙:
1. 반드시 제공된 문서 내용만을 사용하여 답변하세요.
2. 문서에 정보가 없으면 "문서에서 관련 정보를 찾을 수 없습니다"라고 답변하세요.
3. 답변은 명확하고 간결하게 작성하세요.
4. 가능한 경우 문서의 내용을 인용하여 답변하세요.

문서 내용:
{context}

질문: {query}

답변:"""

    # 4. LLM 호출
    print("답변 생성 중...")
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0  # 일관된 답변을 위해 0으로 설정
    )

    response = llm.invoke(prompt)
    answer = response.content

    # 5. 결과 반환
    result = {
        'query': query,
        'answer': answer,
        'retrieved_chunks': retrieved_chunks,
    }

    return result

print("RAG 시스템 준비 완료")

RAG 시스템 준비 완료


## 9. 질의응답 테스트


In [24]:
# 질문 입력
query = "Table 1에서 LiveCode brnch에서 Proprietary Models 중 가장 높은 점수를 받은 모델을 알려줘."  # 원하는 질문으로 변경하세요

# RAG 실행
result = generate_answer(query, top_k=3)

# 결과 출력
print("="*80)
print("[최종 답변]")
print("="*80)
print(result['answer'])

질문: Table 1에서 LiveCode brnch에서 Proprietary Models 중 가장 높은 점수를 받은 모델을 알려줘.

관련 문서 검색 중...
검색 완료: 3개 청크 발견

[1위] 청크 #34 (유사도: 0.4975)
Model MMMU Math- Math- Olympiad AIME LiveCode Live IFEV AL
Vista Vision Bench 24 bench Bench
Proprietary Models
Claude-3.5-Sonnet 70.4 67.7 - - - - - ...

[2위] 청크 #32 (유사도: 0.4227)
73.6% on MMMU) and QvQ-Preview-72B (70.3% vs. 73.6% on MMMU).
Particularly noteworthy is R1V2’s exceptional performance on OlympiadBench, where it ach...

[3위] 청크 #33 (유사도: 0.3708)
74.0% surpasses Claude 3.5 Sonnet (67.7%) and is competitive with Gemini 2 Flash (73.1%) and
Kimi k1.5 longcot (74.9%).
While larger proprietary model...

답변 생성 중...
[최종 답변]
문서에서 관련 정보를 찾을 수 없습니다.
